# MNIST MLP3 — MuonClip polar Jacobian: raw W ESD vs analytic and numerical response spectra

For each matrix checkpoint \(W=U\Sigma V^\top\), this notebook compares three **raw, unnormalized** spectra:

1. the ordinary weight ESD \(\lambda(W^\top W)=\{\sigma_i^2\}\);
2. the exact analytic spectrum of \(G_W=(D\Pi(W))^*D\Pi(W)\), where \(\Pi(W)=UV^\top\);
3. a numerical single-checkpoint finite-difference response Gram spectrum from isotropic random perturbations.

All three are fit by the same in-notebook continuous power-law MLE with a grid search over \(x_{\min}\) minimizing the Kolmogorov--Smirnov distance. No spectral normalization, log binning, KDE, or separate fitting rule is used.

For the analytic polar derivative, with \(A=U^\top E V\),
\[
\Omega_{ij}=\frac{A_{ij}-A_{ji}}{\sigma_i+\sigma_j},\qquad \Omega_{ii}=0.
\]
The nonzero Gram eigenvalues are
\[
\lambda_{ij}^{\rm rot}=\frac{4}{(\sigma_i+\sigma_j)^2},\quad i<j,
\]
and for rectangular matrices
\[
\lambda_i^\perp=\frac{1}{\sigma_i^2}
\]
with multiplicity \(|m-n|\).

For the numerical method, draw isotropic unit-Frobenius directions \(E_a\), compute
\[
X_a=\frac{\Pi(W+\epsilon E_a)-\Pi(W-\epsilon E_a)}{2\epsilon},
\]
stack \(\operatorname{vec}(X_a)\) into \(X\), and diagonalize \(X^\top X\).


In [ ]:
from pathlib import Path
import os,sys,math,random
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from IPython.display import display
import matplotlib.pyplot as plt

ROOT=None
for q in [Path.cwd(),*Path.cwd().parents]:
    if (q/'baseline'/'rg_baselines').is_dir(): ROOT=(q/'baseline').resolve(); break
    if (q/'rg_baselines').is_dir(): ROOT=q.resolve(); break
if ROOT is None: raise RuntimeError('Run from CalculatedContent/rg_optimizers')
sys.path.insert(0,str(ROOT))
from rg_baselines import MLP3,DEFAULT_BASELINE_SEEDS,MNIST_REFERENCE_SUITE_SLUG
from rg_baselines.polar_jacobian import raw_weight_esd,analytic_gram_spectrum,numerical_probe_gram_spectrum,powerlaw_mle_grid_ks,finite_difference_error,probe_convergence

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR=Path(os.environ.get('RG_BASELINE_DATA_DIR',ROOT/'data')).expanduser().resolve()
RUN_ROOT=Path(os.environ.get('RG_BASELINE_RUN_ROOT',ROOT/'runs')).expanduser().resolve()/MNIST_REFERENCE_SUITE_SLUG/'muonclip_polar_jacobian_three_way_mle_ks'; RUN_ROOT.mkdir(parents=True,exist_ok=True)
EPOCHS=int(os.environ.get('MUONCLIP_POLAR_EPOCHS','30')); BATCH=int(os.environ.get('MUONCLIP_POLAR_BATCH','256'))
SEEDS=tuple(int(x) for x in os.environ.get('MUONCLIP_POLAR_SEEDS',','.join(map(str,DEFAULT_BASELINE_SEEDS))).split(','))
PROBES=int(os.environ.get('POLAR_NUMERIC_PROBES','32')); EPS=float(os.environ.get('POLAR_NUMERIC_EPS_REL','1e-5')); NUMERIC_EVERY=int(os.environ.get('POLAR_NUMERIC_EVERY','0'))
MIN_TAIL=int(os.environ.get('POLAR_MLE_KS_MIN_TAIL','5'))
print(DEVICE,RUN_ROOT)


## Power-law estimator

For each candidate tail start \(x_{\min}\) leaving at least `MIN_TAIL` eigenvalues, fit the continuous Pareto exponent
\[
\widehat\alpha=1+\frac{n}{\sum_i\log(x_i/x_{\min})}.
\]
Select the candidate minimizing
\[
D=\sup_x |F_{\rm empirical}(x)-F_{\rm Pareto}(x)|.
\]
The same `powerlaw_mle_grid_ks` function is applied to the raw weight ESD, analytic polar-Gram ESD, and numerical probe-Gram ESD.


In [ ]:
MATRIX_LR=2e-3; AUX_LR=2e-3; MOMENTUM=.95; WEIGHT_DECAY=1e-2; RMS_SCALE=.20; GRAD_CLIP=1.0
@torch.no_grad()
def zeropower(g,steps=5,eps=1e-7):
    t=g.shape[0]>g.shape[1]; x=g.T if t else g; x=x.float()/torch.linalg.vector_norm(x.float()).clamp_min(eps); a,b,c=3.4445,-4.7750,2.0315
    for _ in range(steps): q=x@x.T; x=a*x+(b*q+c*(q@q))@x
    return x.T if t else x
class MuonClip(torch.optim.Optimizer):
    def __init__(self,params): super().__init__(params,dict(lr=MATRIX_LR,momentum=MOMENTUM,weight_decay=WEIGHT_DECAY))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                b=self.state[p].setdefault('momentum_buffer',torch.zeros_like(p.grad)); b.mul_(group['momentum']).add_(p.grad)
                u=zeropower(b).to(p.dtype); u.mul_(RMS_SCALE*math.sqrt(max(p.shape))); p.mul_(1-group['lr']*group['weight_decay']); p.add_(u,alpha=-group['lr'])

rng=np.random.default_rng(123)
for shape in [(3,3),(4,3),(3,5)]:
    W=rng.normal(size=shape); E=rng.normal(size=shape); err=finite_difference_error(W,E)
    print(shape,err); assert err<1e-7


In [ ]:
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3081,))])
full=datasets.MNIST(str(DATA_DIR),train=True,download=True,transform=transform); test=datasets.MNIST(str(DATA_DIR),train=False,download=True,transform=transform)
g=torch.Generator().manual_seed(20_260_807); perm=torch.randperm(len(full),generator=g).tolist(); val_idx,train_idx=perm[:5000],perm[5000:]; train,val=Subset(full,train_idx),Subset(full,val_idx)
def loaders(seed):
    tg=torch.Generator().manual_seed(seed+101); kw=dict(batch_size=BATCH,num_workers=0,pin_memory=DEVICE.type=='cuda')
    return DataLoader(train,shuffle=True,generator=tg,**kw),DataLoader(train,shuffle=False,**kw),DataLoader(val,shuffle=False,**kw),DataLoader(test,shuffle=False,**kw)
@torch.inference_mode()
def evaluate(model,loader):
    model.eval(); loss=correct=n=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); z=model(x); loss+=float(F.cross_entropy(z,y,reduction='sum').cpu()); correct+=int((z.argmax(1)==y).sum().cpu()); n+=y.numel()
    return loss/n,correct/n


In [ ]:
def seed_all(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
def numerical_due(epoch): return epoch in {0,EPOCHS} or (NUMERIC_EVERY>0 and epoch%NUMERIC_EVERY==0)
def fit_row(seed,epoch,layer,method,spectrum,**extra):
    return dict(seed=seed,epoch=epoch,layer=layer,method=method,spectrum_size=len(spectrum),**extra,**powerlaw_mle_grid_ks(spectrum,min_tail=MIN_TAIL))
def analyze(model,seed,epoch):
    rows=[]; spectra={}
    for j,name in enumerate(('fc1','fc2','fc3')):
        W=getattr(model,name).weight.detach().cpu().numpy()
        raw=raw_weight_esd(W); analytic,zero_modes=analytic_gram_spectrum(W)
        rows.append(fit_row(seed,epoch,name,'raw_weight_WtW',raw,zero_modes=np.nan,probes=np.nan,epsilon=np.nan)); spectra[f'raw__seed_{seed}__epoch_{epoch:03d}__{name}']=raw
        rows.append(fit_row(seed,epoch,name,'analytic_polar_JtJ',analytic,zero_modes=zero_modes,probes=np.nan,epsilon=np.nan)); spectra[f'analytic__seed_{seed}__epoch_{epoch:03d}__{name}']=analytic
        if numerical_due(epoch):
            numeric,meta=numerical_probe_gram_spectrum(W,probes=PROBES,eps_rel=EPS,seed=918273+100000*seed+100*epoch+j)
            rows.append(fit_row(seed,epoch,name,'numerical_polar_JtJ',numeric,zero_modes=np.nan,probes=PROBES,epsilon=meta['epsilon'])); spectra[f'numeric__seed_{seed}__epoch_{epoch:03d}__{name}']=numeric
    return rows,spectra

perf=[]; fits=[]; spectra={}
for seed in SEEDS:
    seed_all(seed); tr,tre,va,te=loaders(seed); model=MLP3().to(DEVICE); matrices=[p for p in model.parameters() if p.ndim==2]; biases=[p for p in model.parameters() if p.ndim!=2]; muon=MuonClip(matrices); aux=torch.optim.AdamW(biases,lr=AUX_LR,weight_decay=WEIGHT_DECAY)
    r,s=analyze(model,seed,0); fits+=r; spectra.update(s)
    for epoch in range(1,EPOCHS+1):
        model.train()
        for x,y in tr:
            x,y=x.to(DEVICE),y.to(DEVICE); muon.zero_grad(set_to_none=True); aux.zero_grad(set_to_none=True); loss=F.cross_entropy(model(x),y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); muon.step(); aux.step()
        tl,ta=evaluate(model,tre); vl,vaa=evaluate(model,va); ql,qa=evaluate(model,te); perf.append(dict(seed=seed,epoch=epoch,train_loss=tl,validation_loss=vl,test_loss=ql,train_accuracy=ta,validation_accuracy=vaa,test_accuracy=qa)); r,s=analyze(model,seed,epoch); fits+=r; spectra.update(s); print(seed,epoch,vl)
performance=pd.DataFrame(perf); three_way_fits=pd.DataFrame(fits); performance.to_csv(RUN_ROOT/'performance_by_epoch_and_seed.csv',index=False); three_way_fits.to_csv(RUN_ROOT/'raw_vs_polar_mle_ks_by_epoch_layer_seed.csv',index=False); np.savez_compressed(RUN_ROOT/'raw_vs_polar_spectra.npz',**spectra)


## Three-way exponent comparison

The table below reports \(\alpha\), KS distance \(D\), \(x_{\min}\), and fitted-tail size for every available method. The numerical response spectrum is a finite-probe random-subspace estimate; the analytic spectrum is the full exact positive spectrum.


In [ ]:
cols=['seed','epoch','layer','method','spectrum_size','alpha','D','xmin','tail_evals','probes']
display(three_way_fits[cols].sort_values(['seed','epoch','layer','method']))
alpha_table=three_way_fits.pivot_table(index=['seed','epoch','layer'],columns='method',values='alpha',aggfunc='first').reset_index(); alpha_table.to_csv(RUN_ROOT/'alpha_three_way_comparison.csv',index=False); display(alpha_table)
final=three_way_fits[three_way_fits.epoch.eq(EPOCHS)].copy(); display(final[cols].sort_values(['seed','layer','method']))


## Single-checkpoint numerical convergence

For a selected checkpoint matrix `W`, `probe_convergence` repeats the finite-difference calculation at increasing probe counts and compares its MLE/KS exponent with the analytic polar-Gram exponent.


In [ ]:
# Example after training:
# W=model.fc2.weight.detach().cpu().numpy()
# display(pd.DataFrame(probe_convergence(W,counts=(16,32,64,128,256),eps_rel=EPS,min_tail=MIN_TAIL)))
